In [ ]:
#회계년도 9월 마치는 기업 

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import glob
from datetime import date

# ===== 사용자 입력 =====
COMPANY = "Apple"   # 예: "Apple"
TICKER  = "AAPL"    # 예: "AAPL"
MARKET_PRICE_OVERRIDE = None  # None이면 분기별로 날짜에 맞는 종가 사용

# ==== 회계연도 마감월 (대부분 12, AAPL=9) ====
FY_END_MONTH = 9

# --- DCF 가정 (원하면 조정) ---
WACC = 0.10          # FCFF 할인율
COE  = 0.10          # Owner Earnings 할인율(자기자본비용)
G_SHORT = 0.05       # 1단계 성장률 (N년)
PROJ_YEARS = 5       # 1단계 기간 (년)
G_TERM  = 0.025      # 영구 성장률 (터미널)

# ===== 경로 =====
BASE = Path(r"C:\Users\seung\OneDrive\주식")
BACKTEST_DIR = BASE / "Back Test"
FIN_DIR = BASE / "Financial_Data_real"
SUMMARY_DIR = FIN_DIR / "Summary"
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)

# ===== 가격 CSV 자동 탐색 (Company 폴더 내부) =====
company_folder = BACKTEST_DIR / COMPANY
price_candidates = glob.glob(str(company_folder / f"* {COMPANY} Historical Data.csv"))
if not price_candidates:
    raise FileNotFoundError(f"가격 CSV를 찾을 수 없음: {company_folder}\\* {COMPANY} Historical Data.csv")
PRICE_CSV = price_candidates[0]

# ===== 재무 엑셀 경로 =====
FIN_XLSX = FIN_DIR / f"{TICKER}_financials_Q.xlsx"
if not FIN_XLSX.exists():
    raise FileNotFoundError(f"재무 엑셀 없음: {FIN_XLSX}")

# ------------------------------------------------------------
# 1) 재무 4개 시트 로드 & 가로→세로 변환 (날짜가 행)
# ------------------------------------------------------------
def load_financials(fin_path: Path):
    income = pd.read_excel(fin_path, sheet_name="Income Statement", index_col=0)
    balance = pd.read_excel(fin_path, sheet_name="Balance Sheet", index_col=0)
    cashflw = pd.read_excel(fin_path, sheet_name="Cash Flow Statement", index_col=0)
    ratios  = pd.read_excel(fin_path, sheet_name="Key Financial Ratios", index_col=0)

    def wide_to_long(df):
        df = df.copy()
        df.index.name = "Metric"
        out = df.T.reset_index().rename(columns={"index": "Date"})
        out["Date"] = pd.to_datetime(out["Date"])
        return out

    return wide_to_long(income), wide_to_long(balance), wide_to_long(cashflw), wide_to_long(ratios)

income_q, balance_q, cash_q, ratios_q = load_financials(FIN_XLSX)

# ------------------------------------------------------------
# 2) 분기 데이터 병합 (같은 Date 기준 left join)
# ------------------------------------------------------------
q = income_q.merge(balance_q, on="Date", how="left", suffixes=("", "_BS"))\
            .merge(cash_q, on="Date", how="left", suffixes=("", "_CF"))\
            .merge(ratios_q, on="Date", how="left", suffixes=("", "_RT"))
q.sort_values("Date", inplace=True)

# ------------------------------------------------------------
# 3) 가격 데이터 로드 및 분기말 종가 매칭 (가장 가까운 이전 거래일)
# ------------------------------------------------------------
price = pd.read_csv(PRICE_CSV)
price.rename(columns=lambda c: c.strip(), inplace=True)
price["Date"] = pd.to_datetime(price["Date"])
price.sort_values("Date", inplace=True)

price_col = "Price" if "Price" in price.columns else [c for c in price.columns if "Price" in c][0]
px = price[["Date", price_col]].rename(columns={price_col: "Price"})

q = pd.merge_asof(q.sort_values("Date"), px, on="Date", direction="backward")
if MARKET_PRICE_OVERRIDE is not None:
    q["Price"] = float(MARKET_PRICE_OVERRIDE)

# ------------------------------------------------------------
# 4) 공통 계산 함수들
# ------------------------------------------------------------
def compute_row_metrics(r):
    """분기 스냅샷 기준 핵심 지표 계산"""
    get = lambda col, default=0.0: float(r[col]) if pd.notna(r.get(col)) else default

    revenue = get("Revenue")
    net_income = get("Net Income")
    eps = get("EPS - Earnings Per Share") or get("Basic EPS")
    ebit = get("EBIT")
    ebitda = get("EBITDA")
    shares_out = get("Shares Outstanding") or get("Basic Shares Outstanding")
    total_assets = get("Total Assets")
    total_liabilities = get("Total Liabilities")
    sh_equity = get("Share Holder Equity")
    retained = get("Retained Earnings (Accumulated Deficit)")
    cash = get("Cash On Hand")
    curr_assets = get("Total Current Assets")
    curr_liab = get("Total Current Liabilities")

    dep = get("Total Depreciation And Amortization - Cash Flow")
    capex = -get("Net Change In Property, Plant, And Equipment")      # 지출(+)
    delta_wc_cf = -get("Total Change In Assets/Liabilities")          # 운전자본 증가(+)
    price = get("Price")

    market_cap = price * shares_out if shares_out else np.nan
    book_value_per_share = (sh_equity / shares_out) if shares_out else np.nan

    # NCAV
    ncav = curr_assets - total_liabilities if (curr_assets and total_liabilities) else np.nan
    underval_ncav = (market_cap < ncav) if (pd.notna(market_cap) and pd.notna(ncav)) else np.nan

    # Graham Number
    graham = np.sqrt(22.5 * max(eps, 0) * max(book_value_per_share, 0)) if pd.notna(eps) and pd.notna(book_value_per_share) else np.nan

    # FCFF (Damodaran)
    nopat = ebit * (1 - 0.25) if pd.notna(ebit) else np.nan
    fcff = (nopat + dep - capex - delta_wc_cf) if pd.notna(nopat) else np.nan

    # Owner Earnings
    owner_earn = (net_income + dep - capex - delta_wc_cf) if pd.notna(net_income) else np.nan

    # EVA
    eva = (nopat - total_assets * WACC) if pd.notna(nopat) and pd.notna(total_assets) else np.nan

    # Altman Z
    wc = curr_assets - curr_liab if pd.notna(curr_assets) and pd.notna(curr_liab) else np.nan
    z = (1.2 * (wc / total_assets) + 1.4 * (retained / total_assets) +
         3.3 * (ebit / total_assets) + 0.6 * (market_cap / total_liabilities) +
         1.0 * (revenue / total_assets)) if all(pd.notna(x) for x in [wc, total_assets, retained, ebit, market_cap, total_liabilities, revenue]) else np.nan

    # Magic Formula
    net_fixed_assets = total_assets - curr_assets if pd.notna(total_assets) and pd.notna(curr_assets) else np.nan
    invested_capital = (curr_assets + net_fixed_assets) if pd.notna(net_fixed_assets) else np.nan
    roic = (ebit / invested_capital) if (pd.notna(ebit) and pd.notna(invested_capital) and invested_capital != 0) else np.nan
    ev = (market_cap + total_liabilities - cash) if all(pd.notna(x) for x in [market_cap, total_liabilities, cash]) else np.nan
    ev_ebit = (ev / ebit) if (pd.notna(ev) and pd.notna(ebit) and ebit != 0) else np.nan

    # DDM
    div_paid = abs(get("Common Stock Dividends Paid"))
    div_ps = (div_paid / shares_out) if (shares_out and div_paid) else np.nan
    ddm_val = (div_ps * (1 + G_SHORT) / (COE - G_SHORT)) if pd.notna(div_ps) else np.nan

    return pd.Series({
        "Price": price, "Market Cap": market_cap,
        "NCAV": ncav, "Undervalued by NCAV": underval_ncav,
        "Graham Number": graham, "FCFF": fcff, "Owner Earnings": owner_earn,
        "EVA": eva, "Altman Z": z, "ROIC": roic, "EV/EBIT": ev_ebit, "DDM Value": ddm_val
    })

def dcf_equity_value_from_fcf(fcf0, net_debt, rate, g_short, years, g_term):
    """FCF(FCFF or Owner Earnings)에서 Equity Value 계산 (2단계 성장 + 터미널)."""
    if pd.isna(fcf0) or fcf0 <= 0:
        return np.nan
    pv = 0.0
    for t in range(1, years + 1):
        cf_t = fcf0 * ((1 + g_short) ** t)
        pv += cf_t / ((1 + rate) ** t)
    cf_N = fcf0 * ((1 + g_short) ** years)
    tv = cf_N * (1 + g_term) / (rate - g_term)
    pv_tv = tv / ((1 + rate) ** years)
    ev = pv + pv_tv
    return ev - (net_debt if pd.notna(net_debt) else 0.0)

# 5) 분기별(Q) 계산
q_metrics = q.apply(compute_row_metrics, axis=1)
quarterly = pd.concat([q[["Date"]], q_metrics], axis=1).set_index("Date")

# ------------------------------------------------------------
# 6) 연환산(TTM) 시계열 + PEG/PEGY + DCF 적정가
# ------------------------------------------------------------
qq = q.set_index("Date").sort_index()

def ttm_series(col):
    # 각 분기 시점의 직전 4개 분기 합산 (TTM)
    return qq[col].rolling(4, min_periods=4).sum()

# 흐름형 항목 TTM
revenue_ttm    = ttm_series("Revenue")
net_income_ttm = ttm_series("Net Income")
ebit_ttm       = ttm_series("EBIT")
dep_ttm        = ttm_series("Total Depreciation And Amortization - Cash Flow")
capex_ttm      = (-qq["Net Change In Property, Plant, And Equipment"]).rolling(4, min_periods=4).sum()
delta_wc_ttm   = (-qq["Total Change In Assets/Liabilities"]).rolling(4, min_periods=4).sum()
div_ps_ttm     = (abs(qq["Common Stock Dividends Paid"]) / qq["Shares Outstanding"]).rolling(4, min_periods=4).sum()

# 스톡(스냅샷) 항목은 분기말 값 사용
snap = qq[["Total Assets","Total Liabilities","Share Holder Equity","Total Current Assets",
           "Total Current Liabilities","Cash On Hand","Shares Outstanding","Price"]].copy()

# 기본 TTM 계산
nopat_ttm = ebit_ttm * (1 - 0.25)
fcff_ttm  = nopat_ttm + dep_ttm - capex_ttm - delta_wc_ttm
owner_ttm = net_income_ttm + dep_ttm - capex_ttm - delta_wc_ttm

book_value_per_share = (snap["Share Holder Equity"] / snap["Shares Outstanding"])
market_cap_ttm = snap["Price"] * snap["Shares Outstanding"]
ncav_ttm = snap["Total Current Assets"] - snap["Total Liabilities"]
ev_ttm = market_cap_ttm + snap["Total Liabilities"] - snap["Cash On Hand"]
roic_ttm = ebit_ttm / (snap["Total Current Assets"] + (snap["Total Assets"] - snap["Total Current Assets"]))
ev_ebit_ttm = ev_ttm / ebit_ttm
wc_ttm = snap["Total Current Assets"] - snap["Total Current Liabilities"]
z_ttm = (1.2 * (wc_ttm / snap["Total Assets"]) +
         1.4 * (qq["Retained Earnings (Accumulated Deficit)"] / snap["Total Assets"]) +
         3.3 * (ebit_ttm / snap["Total Assets"]) +
         0.6 * (market_cap_ttm / snap["Total Liabilities"]) +
         1.0 * (revenue_ttm / snap["Total Assets"]))

# PEG / PEGY (안전한 분모 사용)
eps_ttm = net_income_ttm / snap["Shares Outstanding"]
pe_ttm = snap["Price"] / eps_ttm
eps_growth_yoy = eps_ttm.pct_change(4)              # 전년 동분기 대비 TTM EPS 성장률 (소수)
div_yield_ttm  = div_ps_ttm / snap["Price"]         # 소수

def safe_div(n, d):
    return n / d.where(d.abs() > 1e-9)  # |d|가 아주 작으면 NaN

peg_ttm  = safe_div(pe_ttm, (eps_growth_yoy * 100))                     # 성장률 % 기준
pegy_ttm = safe_div(pe_ttm, (eps_growth_yoy*100) + (div_yield_ttm*100)) # 배당 포함

# --- DCF 적정가 (TTM 현금흐름 이용) ---
net_debt_series = snap["Total Liabilities"] - snap["Cash On Hand"]
equity_from_fcff  = pd.Series({dt: dcf_equity_value_from_fcf(fcff_ttm.loc[dt], net_debt_series.loc[dt], WACC, G_SHORT, PROJ_YEARS, G_TERM) for dt in qq.index})
equity_from_owner = pd.Series({dt: dcf_equity_value_from_fcf(owner_ttm.loc[dt], 0.0, COE, G_SHORT, PROJ_YEARS, G_TERM) for dt in qq.index})

dcf_price_fcff  = equity_from_fcff  / snap["Shares Outstanding"]
dcf_price_owner = equity_from_owner / snap["Shares Outstanding"]

# ---- 모든 분기별 TTM 표 (그대로 유지) ----
yearly_ttm_full = pd.DataFrame({
    "Price": snap["Price"], "Market Cap": market_cap_ttm, "NCAV": ncav_ttm,
    "Undervalued by NCAV": market_cap_ttm < ncav_ttm,
    "Graham Number": np.sqrt(22.5 * (eps_ttm) * book_value_per_share),
    "FCFF": fcff_ttm, "Owner Earnings": owner_ttm, "EVA": (nopat_ttm - snap["Total Assets"] * WACC),
    "Altman Z": z_ttm, "ROIC": roic_ttm, "EV/EBIT": ev_ebit_ttm,
    "DDM Value": (div_ps_ttm * (1 + G_SHORT) / (COE - G_SHORT)),
    "EPS_TTM": eps_ttm, "PE_TTM": pe_ttm, "EPS_Growth_YoY_%": eps_growth_yoy * 100,
    "PEG_TTM": peg_ttm, "PEGY_TTM": pegy_ttm,
    "DCF_FCFF_Price": dcf_price_fcff, "DCF_OwnerEarnings_Price": dcf_price_owner
})

# ---- '연간 스냅샷'을 회계연도 Q4(마감 분기) 시점의 TTM으로 선택 ----
freq_map = {12:'Q-DEC',11:'Q-NOV',10:'Q-OCT',9:'Q-SEP',8:'Q-AUG',7:'Q-JUL',6:'Q-JUN',5:'Q-MAY',4:'Q-APR',3:'Q-MAR',2:'Q-FEB',1:'Q-JAN'}
fq_period = yearly_ttm_full.index.to_period(freq_map[FY_END_MONTH])
FQ = fq_period.quarter  # 1~4
yearly_snapshot = yearly_ttm_full[(FQ == 4)].copy()

# ------------------------------------------------------------
# 6.5) 분기 기준 PEG (QoQ 성장률 기반)
# ------------------------------------------------------------

eps_q = qq["EPS - Earnings Per Share"].copy()
eps_q_growth_qoq = eps_q.pct_change(1) * 100  # QoQ EPS 성장률 (%)

price_q = qq["Price"]
pe_q = price_q / eps_q

div_q = abs(qq["Common Stock Dividends Paid"])
shares_out_q = qq["Shares Outstanding"]
div_ps_q = div_q / shares_out_q
div_yield_q = div_ps_q / price_q * 100  # %

def safe_div_q(n, d):
    return n / d.where(d.abs() > 1e-9)

peg_q = safe_div_q(pe_q, eps_q_growth_qoq)
pegy_q = safe_div_q(pe_q, eps_q_growth_qoq + div_yield_q)

# ------------------------------------------------------------
# 6.6) quarterly에 값 추가
# ------------------------------------------------------------

quarterly["EPS_Q"] = eps_q
quarterly["PE_Q"] = pe_q
quarterly["EPS_Growth_QoQ_%"] = eps_q_growth_qoq
quarterly["PEG_Q"] = peg_q
quarterly["PEGY_Q"] = pegy_q


# ------------------------------------------------------------
# 7) 엑셀 저장
# ------------------------------------------------------------
today_str = date.today().isoformat()
out_path = SUMMARY_DIR / f"{today_str}_{TICKER}_{COMPANY}_stock_summary.xlsx"

with pd.ExcelWriter(out_path, engine="xlsxwriter") as writer:
    quarterly.round(4).to_excel(writer, sheet_name="Quarterly")
    yearly_snapshot.round(4).to_excel(writer, sheet_name="Yearly_TTM")
    q.to_excel(writer, sheet_name="Inputs (Merged Q)")
print(f"✅ 저장 완료: {out_path}")


In [ ]:
# 회계년도 12월에 마치는 기업

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import glob
from datetime import date

# ===== 사용자 입력 =====
COMPANY = "Apple"   # 예: "Apple"
TICKER  = "AAPL"    # 예: "AAPL"
MARKET_PRICE_OVERRIDE = None  # None이면 분기별로 날짜에 맞는 종가 사용

# ==== 회계연도 마감월 (대부분 12, AAPL=9) ====
FY_END_MONTH = 9

# --- DCF 가정 (원하면 조정) ---
WACC = 0.10          # FCFF 할인율
COE  = 0.10          # Owner Earnings 할인율(자기자본비용)
G_SHORT = 0.05       # 1단계 성장률 (N년)
PROJ_YEARS = 5       # 1단계 기간 (년)
G_TERM  = 0.025      # 영구 성장률 (터미널)

# ===== 경로 =====
BASE = Path(r"C:\Users\seung\OneDrive\주식")
BACKTEST_DIR = BASE / "Back Test"
FIN_DIR = BASE / "Financial_Data_real"
SUMMARY_DIR = FIN_DIR / "Summary"
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)

# ===== 가격 CSV 자동 탐색 (Company 폴더 내부) =====
company_folder = BACKTEST_DIR / COMPANY
price_candidates = glob.glob(str(company_folder / f"* {COMPANY} Historical Data.csv"))
if not price_candidates:
    raise FileNotFoundError(f"가격 CSV를 찾을 수 없음: {company_folder}\\* {COMPANY} Historical Data.csv")
PRICE_CSV = price_candidates[0]

# ===== 재무 엑셀 경로 =====
FIN_XLSX = FIN_DIR / f"{TICKER}_financials_Q.xlsx"
if not FIN_XLSX.exists():
    raise FileNotFoundError(f"재무 엑셀 없음: {FIN_XLSX}")

# ------------------------------------------------------------
# 1) 재무 4개 시트 로드 & 가로→세로 변환 (날짜가 행)
# ------------------------------------------------------------
def load_financials(fin_path: Path):
    income = pd.read_excel(fin_path, sheet_name="Income Statement", index_col=0)
    balance = pd.read_excel(fin_path, sheet_name="Balance Sheet", index_col=0)
    cashflw = pd.read_excel(fin_path, sheet_name="Cash Flow Statement", index_col=0)
    ratios  = pd.read_excel(fin_path, sheet_name="Key Financial Ratios", index_col=0)

    def wide_to_long(df):
        df = df.copy()
        df.index.name = "Metric"
        out = df.T.reset_index().rename(columns={"index": "Date"})
        out["Date"] = pd.to_datetime(out["Date"])
        return out

    return wide_to_long(income), wide_to_long(balance), wide_to_long(cashflw), wide_to_long(ratios)

income_q, balance_q, cash_q, ratios_q = load_financials(FIN_XLSX)

# ------------------------------------------------------------
# 2) 분기 데이터 병합 (같은 Date 기준 left join)
# ------------------------------------------------------------
q = income_q.merge(balance_q, on="Date", how="left", suffixes=("", "_BS"))\
            .merge(cash_q, on="Date", how="left", suffixes=("", "_CF"))\
            .merge(ratios_q, on="Date", how="left", suffixes=("", "_RT"))
q.sort_values("Date", inplace=True)

# ------------------------------------------------------------
# 3) 가격 데이터 로드 및 분기말 종가 매칭 (가장 가까운 이전 거래일)
# ------------------------------------------------------------
price = pd.read_csv(PRICE_CSV)
price.rename(columns=lambda c: c.strip(), inplace=True)
price["Date"] = pd.to_datetime(price["Date"])
price.sort_values("Date", inplace=True)

price_col = "Price" if "Price" in price.columns else [c for c in price.columns if "Price" in c][0]
px = price[["Date", price_col]].rename(columns={price_col: "Price"})

q = pd.merge_asof(q.sort_values("Date"), px, on="Date", direction="backward")
if MARKET_PRICE_OVERRIDE is not None:
    q["Price"] = float(MARKET_PRICE_OVERRIDE)

# ------------------------------------------------------------
# 4) 공통 계산 함수들
# ------------------------------------------------------------
def compute_row_metrics(r):
    """분기 스냅샷 기준 핵심 지표 계산"""
    get = lambda col, default=0.0: float(r[col]) if pd.notna(r.get(col)) else default

    revenue = get("Revenue")
    net_income = get("Net Income")
    eps = get("EPS - Earnings Per Share") or get("Basic EPS")
    ebit = get("EBIT")
    ebitda = get("EBITDA")
    shares_out = get("Shares Outstanding") or get("Basic Shares Outstanding")
    total_assets = get("Total Assets")
    total_liabilities = get("Total Liabilities")
    sh_equity = get("Share Holder Equity")
    retained = get("Retained Earnings (Accumulated Deficit)")
    cash = get("Cash On Hand")
    curr_assets = get("Total Current Assets")
    curr_liab = get("Total Current Liabilities")

    dep = get("Total Depreciation And Amortization - Cash Flow")
    capex = -get("Net Change In Property, Plant, And Equipment")      # 지출(+)
    delta_wc_cf = -get("Total Change In Assets/Liabilities")          # 운전자본 증가(+)
    price = get("Price")

    market_cap = price * shares_out if shares_out else np.nan
    book_value_per_share = (sh_equity / shares_out) if shares_out else np.nan

    # NCAV
    ncav = curr_assets - total_liabilities if (curr_assets and total_liabilities) else np.nan
    underval_ncav = (market_cap < ncav) if (pd.notna(market_cap) and pd.notna(ncav)) else np.nan

    # Graham Number
    graham = np.sqrt(22.5 * max(eps, 0) * max(book_value_per_share, 0)) if pd.notna(eps) and pd.notna(book_value_per_share) else np.nan

    # FCFF (Damodaran)
    nopat = ebit * (1 - 0.25) if pd.notna(ebit) else np.nan
    fcff = (nopat + dep - capex - delta_wc_cf) if pd.notna(nopat) else np.nan

    # Owner Earnings
    owner_earn = (net_income + dep - capex - delta_wc_cf) if pd.notna(net_income) else np.nan

    # EVA
    eva = (nopat - total_assets * WACC) if pd.notna(nopat) and pd.notna(total_assets) else np.nan

    # Altman Z
    wc = curr_assets - curr_liab if pd.notna(curr_assets) and pd.notna(curr_liab) else np.nan
    z = (1.2 * (wc / total_assets) + 1.4 * (retained / total_assets) +
         3.3 * (ebit / total_assets) + 0.6 * (market_cap / total_liabilities) +
         1.0 * (revenue / total_assets)) if all(pd.notna(x) for x in [wc, total_assets, retained, ebit, market_cap, total_liabilities, revenue]) else np.nan

    # Magic Formula
    net_fixed_assets = total_assets - curr_assets if pd.notna(total_assets) and pd.notna(curr_assets) else np.nan
    invested_capital = (curr_assets + net_fixed_assets) if pd.notna(net_fixed_assets) else np.nan
    roic = (ebit / invested_capital) if (pd.notna(ebit) and pd.notna(invested_capital) and invested_capital != 0) else np.nan
    ev = (market_cap + total_liabilities - cash) if all(pd.notna(x) for x in [market_cap, total_liabilities, cash]) else np.nan
    ev_ebit = (ev / ebit) if (pd.notna(ev) and pd.notna(ebit) and ebit != 0) else np.nan

    # DDM
    div_paid = abs(get("Common Stock Dividends Paid"))
    div_ps = (div_paid / shares_out) if (shares_out and div_paid) else np.nan
    ddm_val = (div_ps * (1 + G_SHORT) / (COE - G_SHORT)) if pd.notna(div_ps) else np.nan

    return pd.Series({
        "Price": price, "Market Cap": market_cap,
        "NCAV": ncav, "Undervalued by NCAV": underval_ncav,
        "Graham Number": graham, "FCFF": fcff, "Owner Earnings": owner_earn,
        "EVA": eva, "Altman Z": z, "ROIC": roic, "EV/EBIT": ev_ebit, "DDM Value": ddm_val
    })

def dcf_equity_value_from_fcf(fcf0, net_debt, rate, g_short, years, g_term):
    """FCF(FCFF or Owner Earnings)에서 Equity Value 계산 (2단계 성장 + 터미널)."""
    if pd.isna(fcf0) or fcf0 <= 0:
        return np.nan
    pv = 0.0
    for t in range(1, years + 1):
        cf_t = fcf0 * ((1 + g_short) ** t)
        pv += cf_t / ((1 + rate) ** t)
    cf_N = fcf0 * ((1 + g_short) ** years)
    tv = cf_N * (1 + g_term) / (rate - g_term)
    pv_tv = tv / ((1 + rate) ** years)
    ev = pv + pv_tv
    return ev - (net_debt if pd.notna(net_debt) else 0.0)

# 5) 분기별(Q) 계산
q_metrics = q.apply(compute_row_metrics, axis=1)
quarterly = pd.concat([q[["Date"]], q_metrics], axis=1).set_index("Date")

# ------------------------------------------------------------
# 6) 연환산(TTM) 시계열 + PEG/PEGY + DCF 적정가
# ------------------------------------------------------------
qq = q.set_index("Date").sort_index()

def ttm_series(col):
    # 각 분기 시점의 직전 4개 분기 합산 (TTM)
    return qq[col].rolling(4, min_periods=4).sum()

# 흐름형 항목 TTM
revenue_ttm    = ttm_series("Revenue")
net_income_ttm = ttm_series("Net Income")
ebit_ttm       = ttm_series("EBIT")
dep_ttm        = ttm_series("Total Depreciation And Amortization - Cash Flow")
capex_ttm      = (-qq["Net Change In Property, Plant, And Equipment"]).rolling(4, min_periods=4).sum()
delta_wc_ttm   = (-qq["Total Change In Assets/Liabilities"]).rolling(4, min_periods=4).sum()
div_ps_ttm     = (abs(qq["Common Stock Dividends Paid"]) / qq["Shares Outstanding"]).rolling(4, min_periods=4).sum()

# 스톡(스냅샷) 항목은 분기말 값 사용
snap = qq[["Total Assets","Total Liabilities","Share Holder Equity","Total Current Assets",
           "Total Current Liabilities","Cash On Hand","Shares Outstanding","Price"]].copy()

# 기본 TTM 계산
nopat_ttm = ebit_ttm * (1 - 0.25)
fcff_ttm  = nopat_ttm + dep_ttm - capex_ttm - delta_wc_ttm
owner_ttm = net_income_ttm + dep_ttm - capex_ttm - delta_wc_ttm

book_value_per_share = (snap["Share Holder Equity"] / snap["Shares Outstanding"])
market_cap_ttm = snap["Price"] * snap["Shares Outstanding"]
ncav_ttm = snap["Total Current Assets"] - snap["Total Liabilities"]
ev_ttm = market_cap_ttm + snap["Total Liabilities"] - snap["Cash On Hand"]
roic_ttm = ebit_ttm / (snap["Total Current Assets"] + (snap["Total Assets"] - snap["Total Current Assets"]))
ev_ebit_ttm = ev_ttm / ebit_ttm
wc_ttm = snap["Total Current Assets"] - snap["Total Current Liabilities"]
z_ttm = (1.2 * (wc_ttm / snap["Total Assets"]) +
         1.4 * (qq["Retained Earnings (Accumulated Deficit)"] / snap["Total Assets"]) +
         3.3 * (ebit_ttm / snap["Total Assets"]) +
         0.6 * (market_cap_ttm / snap["Total Liabilities"]) +
         1.0 * (revenue_ttm / snap["Total Assets"]))

# PEG / PEGY (안전한 분모 사용)
eps_ttm = net_income_ttm / snap["Shares Outstanding"]
pe_ttm = snap["Price"] / eps_ttm
eps_growth_yoy = eps_ttm.pct_change(4)              # 전년 동분기 대비 TTM EPS 성장률 (소수)
div_yield_ttm  = div_ps_ttm / snap["Price"]         # 소수

def safe_div(n, d):
    return n / d.where(d.abs() > 1e-9)  # |d|가 아주 작으면 NaN

peg_ttm  = safe_div(pe_ttm, (eps_growth_yoy * 100))                     # 성장률 % 기준
pegy_ttm = safe_div(pe_ttm, (eps_growth_yoy*100) + (div_yield_ttm*100)) # 배당 포함

# --- DCF 적정가 (TTM 현금흐름 이용) ---
net_debt_series = snap["Total Liabilities"] - snap["Cash On Hand"]
equity_from_fcff  = pd.Series({dt: dcf_equity_value_from_fcf(fcff_ttm.loc[dt], net_debt_series.loc[dt], WACC, G_SHORT, PROJ_YEARS, G_TERM) for dt in qq.index})
equity_from_owner = pd.Series({dt: dcf_equity_value_from_fcf(owner_ttm.loc[dt], 0.0, COE, G_SHORT, PROJ_YEARS, G_TERM) for dt in qq.index})

dcf_price_fcff  = equity_from_fcff  / snap["Shares Outstanding"]
dcf_price_owner = equity_from_owner / snap["Shares Outstanding"]

# ---- 모든 분기별 TTM 표 (그대로 유지) ----
yearly_ttm_full = pd.DataFrame({
    "Price": snap["Price"], "Market Cap": market_cap_ttm, "NCAV": ncav_ttm,
    "Undervalued by NCAV": market_cap_ttm < ncav_ttm,
    "Graham Number": np.sqrt(22.5 * (eps_ttm) * book_value_per_share),
    "FCFF": fcff_ttm, "Owner Earnings": owner_ttm, "EVA": (nopat_ttm - snap["Total Assets"] * WACC),
    "Altman Z": z_ttm, "ROIC": roic_ttm, "EV/EBIT": ev_ebit_ttm,
    "DDM Value": (div_ps_ttm * (1 + G_SHORT) / (COE - G_SHORT)),
    "EPS_TTM": eps_ttm, "PE_TTM": pe_ttm, "EPS_Growth_YoY_%": eps_growth_yoy * 100,
    "PEG_TTM": peg_ttm, "PEGY_TTM": pegy_ttm,
    "DCF_FCFF_Price": dcf_price_fcff, "DCF_OwnerEarnings_Price": dcf_price_owner
})

# ---- '연간 스냅샷'을 회계연도 Q4(마감 분기) 시점의 TTM으로 선택 ----
freq_map = {12:'Q-DEC',11:'Q-NOV',10:'Q-OCT',9:'Q-SEP',8:'Q-AUG',7:'Q-JUL',6:'Q-JUN',5:'Q-MAY',4:'Q-APR',3:'Q-MAR',2:'Q-FEB',1:'Q-JAN'}
fq_period = yearly_ttm_full.index.to_period(freq_map[FY_END_MONTH])
FQ = fq_period.quarter  # 1~4
yearly_snapshot = yearly_ttm_full[(FQ == 4)].copy()

# ------------------------------------------------------------
# 6.5) 분기 기준 PEG (QoQ 성장률 기반)
# ------------------------------------------------------------

eps_q = qq["EPS - Earnings Per Share"].copy()
eps_q_growth_qoq = eps_q.pct_change(1) * 100  # QoQ EPS 성장률 (%)

price_q = qq["Price"]
pe_q = price_q / eps_q

div_q = abs(qq["Common Stock Dividends Paid"])
shares_out_q = qq["Shares Outstanding"]
div_ps_q = div_q / shares_out_q
div_yield_q = div_ps_q / price_q * 100  # %

def safe_div_q(n, d):
    return n / d.where(d.abs() > 1e-9)

peg_q = safe_div_q(pe_q, eps_q_growth_qoq)
pegy_q = safe_div_q(pe_q, eps_q_growth_qoq + div_yield_q)

# ------------------------------------------------------------
# 6.6) quarterly에 값 추가
# ------------------------------------------------------------

quarterly["EPS_Q"] = eps_q
quarterly["PE_Q"] = pe_q
quarterly["EPS_Growth_QoQ_%"] = eps_q_growth_qoq
quarterly["PEG_Q"] = peg_q
quarterly["PEGY_Q"] = pegy_q


# ------------------------------------------------------------
# 7) 엑셀 저장
# ------------------------------------------------------------
today_str = date.today().isoformat()
out_path = SUMMARY_DIR / f"{today_str}_{TICKER}_{COMPANY}_stock_summary.xlsx"

with pd.ExcelWriter(out_path, engine="xlsxwriter") as writer:
    quarterly.round(4).to_excel(writer, sheet_name="Quarterly")
    yearly_snapshot.round(4).to_excel(writer, sheet_name="Yearly_TTM")
    q.to_excel(writer, sheet_name="Inputs (Merged Q)")
print(f"✅ 저장 완료: {out_path}")


In [ ]:
# 그냥 다 할 수 있는것

In [ ]:
# import pandas as pd
# import numpy as np
# from pathlib import Path
# import glob
# from datetime import date

# # ====== 설정: 분석할 종목 목록 ======
# companies = {
#     "GOOG": ("alphabet", 12),
#     "BN": ("brookfield", 12),
#     "OXY": ("occidental petroleum", 12),
#     "CRSP": ("crispr therapeutics ag", 12),
#     "NVO": ("novo nordisk", 12),
#     "PATH": ("uipath", 12),
#     "OKTA": ("okta", 12),
#     "ILMN": ("illumina", 12),
#     "TSM": ("taiwan semiconductor manufacturing", 12),
#     "NTRA": ("natera", 12),
#     "META": ("meta platforms", 12),
#     "NU": ("nu holdings", 12),
#     "CPNG": ("coupang", 12),
#     "CVX": ("chevron", 12),
#     "PLTR": ("palantir technologies", 12),
#     "DDOG": ("datadog", 12),
#     "CRWD": ("crowdstrike", 12),
#     "AVGO": ("broadcom", 12),
#     "UNH": ("unitedhealth group", 12),
#     "LMND": ("lemonade", 12),
#     "AAPL": ("apple", 9),
#     "MITK": ("mitek systems", 12),
#     "MSFT": ("microsoft", 9),
#     "BBAI": ("bigbearai holdings", 12),
#     "VST": ("vistra", 12),
#     "VRT": ("vertiv holdings", 12),
#     "MP": ("mp materials", 12),
#     "COST": ("costco", 12),
#     "TEVA": ("teva pharmaceutical industries", 12),
#     "TSLA": ("tesla", 12),
#     "U": ("unity software", 12),
#     "RDDT": ("reddit", 12),
#     "SNOW": ("snowflake", 12),
#     "RKLB": ("rocket lab", 12),
#     "MELI": ("mercadolibre", 12),
#     "EH": ("ehang holdings", 12),
#     "GRAL": ("grail", 12),
#     "PLUG": ("plug power", 12),
#     "SE": ("sea", 12),
#     "TEM": ("tempus ai", 12),
#     "IREN": ("iren", 12),
#     "PI": ("impinj", 12),
#     "JOBY": ("joby aviation", 12),
#     "APP": ("applovin", 12),
#     "NVDA": ("nvidia", 1),
#     "COIN": ("coinbase gloabl", 12)
# }

# # ====== DCF 공통 가정 ======
# WACC = 0.10
# COE = 0.10
# G_SHORT = 0.05
# PROJ_YEARS = 5
# G_TERM = 0.025

# # ====== 기본 경로 ======
# BASE = Path(r"C:\Users\seung\OneDrive\주식")
# BACKTEST_DIR = BASE / "Back Test"
# FIN_DIR = BASE / "Financial_Data_real"
# SUMMARY_DIR = FIN_DIR / "Summary"
# SUMMARY_DIR.mkdir(parents=True, exist_ok=True)


# # ====== 개별 기업 분석 함수 ======
# def analyze_company(TICKER, COMPANY, FY_END_MONTH, MARKET_PRICE_OVERRIDE=None):
#     print(f"▶ 분석 시작: {TICKER} ({COMPANY})")

#     company_folder = BACKTEST_DIR / COMPANY
#     price_candidates = glob.glob(str(company_folder / f"* {COMPANY} Historical Data.csv"))
#     if not price_candidates:
#         print(f"❌ 가격 CSV를 찾을 수 없음: {company_folder}\\* {COMPANY} Historical Data.csv")
#         return
#     PRICE_CSV = price_candidates[0]

#     FIN_XLSX = FIN_DIR / f"{TICKER}_financials_Q.xlsx"
#     if not FIN_XLSX.exists():
#         print(f"❌ 재무 엑셀 없음: {FIN_XLSX}")
#         return

#     # --- 재무 시트 로드 ---
#     def load_financials(fin_path: Path):
#         income = pd.read_excel(fin_path, sheet_name="Income Statement", index_col=0)
#         balance = pd.read_excel(fin_path, sheet_name="Balance Sheet", index_col=0)
#         cashflw = pd.read_excel(fin_path, sheet_name="Cash Flow Statement", index_col=0)
#         ratios  = pd.read_excel(fin_path, sheet_name="Key Financial Ratios", index_col=0)

#         def wide_to_long(df):
#             df = df.copy()
#             df.index.name = "Metric"
#             out = df.T.reset_index().rename(columns={"index": "Date"})
#             out["Date"] = pd.to_datetime(out["Date"])
#             return out

#         return wide_to_long(income), wide_to_long(balance), wide_to_long(cashflw), wide_to_long(ratios)

#     income_q, balance_q, cash_q, ratios_q = load_financials(FIN_XLSX)

#     q = income_q.merge(balance_q, on="Date", how="left")\
#                 .merge(cash_q, on="Date", how="left")\
#                 .merge(ratios_q, on="Date", how="left")
#     q.sort_values("Date", inplace=True)

#     price = pd.read_csv(PRICE_CSV)
#     price.rename(columns=lambda c: c.strip(), inplace=True)
#     price["Date"] = pd.to_datetime(price["Date"])
#     price.sort_values("Date", inplace=True)

#     price_col = "Price" if "Price" in price.columns else [c for c in price.columns if "Price" in c][0]
#     px = price[["Date", price_col]].rename(columns={price_col: "Price"})

#     q = pd.merge_asof(q.sort_values("Date"), px, on="Date", direction="backward")
#     if MARKET_PRICE_OVERRIDE is not None:
#         q["Price"] = float(MARKET_PRICE_OVERRIDE)

#     def compute_row_metrics(r):
#         get = lambda col, default=0.0: float(r[col]) if pd.notna(r.get(col)) else default
#         eps = get("EPS - Earnings Per Share") or get("Basic EPS")
#         ebit = get("EBIT")
#         net_income = get("Net Income")
#         dep = get("Total Depreciation And Amortization - Cash Flow")
#         capex = -get("Net Change In Property, Plant, And Equipment")
#         delta_wc_cf = -get("Total Change In Assets/Liabilities")
#         price = get("Price")
#         shares_out = get("Shares Outstanding") or get("Basic Shares Outstanding")
#         market_cap = price * shares_out if shares_out else np.nan
#         cash = get("Cash On Hand")
#         total_liabilities = get("Total Liabilities")
#         fcff = (ebit * (1 - 0.25) + dep - capex - delta_wc_cf) if pd.notna(ebit) else np.nan
#         owner_earn = (net_income + dep - capex - delta_wc_cf) if pd.notna(net_income) else np.nan
#         return pd.Series({
#             "Price": price, "Market Cap": market_cap,
#             "FCFF": fcff, "Owner Earnings": owner_earn
#         })

#     q_metrics = q.apply(compute_row_metrics, axis=1)
#     quarterly = pd.concat([q[["Date"]], q_metrics], axis=1).set_index("Date")

#     qq = q.set_index("Date").sort_index()
#     def ttm_series(col): return qq[col].rolling(4, min_periods=4).sum()

#     net_income_ttm = ttm_series("Net Income")
#     ebit_ttm = ttm_series("EBIT")
#     dep_ttm = ttm_series("Total Depreciation And Amortization - Cash Flow")
#     capex_ttm = (-qq["Net Change In Property, Plant, And Equipment"]).rolling(4, min_periods=4).sum()
#     delta_wc_ttm = (-qq["Total Change In Assets/Liabilities"]).rolling(4, min_periods=4).sum()
#     div_ps_ttm = (abs(qq["Common Stock Dividends Paid"]) / qq["Shares Outstanding"]).rolling(4, min_periods=4).sum()

#     snap = qq[["Total Liabilities","Cash On Hand","Shares Outstanding","Price"]].copy()
#     fcff_ttm = ebit_ttm * (1 - 0.25) + dep_ttm - capex_ttm - delta_wc_ttm
#     owner_ttm = net_income_ttm + dep_ttm - capex_ttm - delta_wc_ttm
#     eps_ttm = net_income_ttm / snap["Shares Outstanding"]
#     pe_ttm = snap["Price"] / eps_ttm
#     eps_growth_yoy = eps_ttm.pct_change(4) * 100
#     div_yield_ttm = div_ps_ttm / snap["Price"] * 100

#     safe_div = lambda n, d: n / d.where(d.abs() > 1e-9)
#     peg_ttm = safe_div(pe_ttm, eps_growth_yoy)
#     pegy_ttm = safe_div(pe_ttm, eps_growth_yoy + div_yield_ttm)

#     def dcf_equity_value_from_fcf(fcf0, net_debt, rate, g_short, years, g_term):
#         if pd.isna(fcf0) or fcf0 <= 0: return np.nan
#         pv = sum(fcf0 * (1 + g_short)**t / (1 + rate)**t for t in range(1, years + 1))
#         cf_N = fcf0 * ((1 + g_short) ** years)
#         tv = cf_N * (1 + g_term) / (rate - g_term)
#         pv_tv = tv / ((1 + rate) ** years)
#         return pv + pv_tv - (net_debt if pd.notna(net_debt) else 0.0)

#     net_debt_series = snap["Total Liabilities"] - snap["Cash On Hand"]
#     equity_from_fcff = pd.Series({dt: dcf_equity_value_from_fcf(fcff_ttm.loc[dt], net_debt_series.loc[dt], WACC, G_SHORT, PROJ_YEARS, G_TERM) for dt in qq.index})
#     equity_from_owner = pd.Series({dt: dcf_equity_value_from_fcf(owner_ttm.loc[dt], 0.0, COE, G_SHORT, PROJ_YEARS, G_TERM) for dt in qq.index})
#     dcf_price_fcff = equity_from_fcff / snap["Shares Outstanding"]
#     dcf_price_owner = equity_from_owner / snap["Shares Outstanding"]

#     yearly_ttm_full = pd.DataFrame({
#         "Price": snap["Price"],
#         "EPS_TTM": eps_ttm, "PE_TTM": pe_ttm, "EPS_Growth_YoY_%": eps_growth_yoy,
#         "PEG_TTM": peg_ttm, "PEGY_TTM": pegy_ttm,
#         "DCF_FCFF_Price": dcf_price_fcff,
#         "DCF_OwnerEarnings_Price": dcf_price_owner
#     })

#     # EPS QoQ 성장률 (적자 -> 적자 개선도 양수로 표시)
#     eps_q = qq["EPS - Earnings Per Share"]
    
#     def eps_growth_qoq(series):
#         growth = []
#         for prev, curr in zip(series.shift(1), series):
#             if pd.isna(prev):
#                 growth.append(np.nan)
#             else:
#                 growth.append((curr - prev) / abs(prev) * 100)
#         return pd.Series(growth, index=series.index)

#         import pandas as pd

#     def safe_div(n, d, eps=1e-9):
#         """안전한 나눗셈: |d|<eps 이거나 NaN이면 NaN 반환"""
#         d_safe = d.copy()
#         d_safe = d_safe.where(d_safe.abs() > eps)
#         return n / d_safe
    
#     def eps_growth_qoq(series, method="classic", eps=1e-9):
#         """
#         EPS QoQ 성장률(%)
#         - method="classic": (curr - prev) / prev * 100, 단 prev≈0이면 NaN
#         - method="midpoint": (curr - prev) / ((|curr|+|prev|)/2) * 100  ← prev=0/음수에도 정의 가능
#         - method="log": ln(curr/prev)*100, 단 curr<=0 or prev<=0이면 NaN
#         """
#         series = pd.Series(series).astype(float)
#         prev = series.shift(1)
    
#         if method == "classic":
#             growth = safe_div(series - prev, prev, eps) * 100.0
    
#         elif method == "midpoint":
#             # ‘대칭 성장률’(중간값 분모) – 분모가 0인 극단 케이스도 보호
#             denom = (series.abs() + prev.abs()) / 2.0
#             growth = safe_div(series - prev, denom, eps) * 100.0
    
#         elif method == "log":
#             # 로그성장률은 <=0 값에서 정의되지 않음
#             mask_valid = (series > 0) & (prev > 0)
#             growth = pd.Series(np.nan, index=series.index, dtype=float)
#             growth[mask_valid] = (np.log(series[mask_valid] / prev[mask_valid])) * 100.0
    
#         else:
#             raise ValueError("method must be 'classic', 'midpoint', or 'log'.")
    
#         return growth
    
#     eps_q_growth_qoq = eps_growth_qoq(eps_q)
#     price_q = qq["Price"]
#     pe_q = price_q / eps_q
#     div_q = abs(qq["Common Stock Dividends Paid"])
#     shares_out_q = qq["Shares Outstanding"]
#     div_ps_q = div_q / shares_out_q
#     div_yield_q = div_ps_q / price_q * 100
#     peg_q = safe_div(pe_q, eps_q_growth_qoq)
#     pegy_q = safe_div(pe_q, eps_q_growth_qoq + div_yield_q)

#     quarterly["EPS_Q"] = eps_q
#     quarterly["PE_Q"] = pe_q
#     quarterly["EPS_Growth_QoQ_%"] = eps_q_growth_qoq
#     quarterly["PEG_Q"] = peg_q
#     quarterly["PEGY_Q"] = pegy_q

#     # 회계연도 마지막 분기만 필터링
#     freq_map = {12:'Q-DEC',11:'Q-NOV',10:'Q-OCT',9:'Q-SEP',8:'Q-AUG',7:'Q-JUL',6:'Q-JUN',5:'Q-MAY',4:'Q-APR',3:'Q-MAR',2:'Q-FEB',1:'Q-JAN'}
#     fq_period = yearly_ttm_full.index.to_period(freq_map[FY_END_MONTH])
#     yearly_snapshot = yearly_ttm_full[(fq_period.quarter == 4)].copy()

#     today_str = date.today().isoformat()
#     out_path = SUMMARY_DIR / f"{today_str}_{TICKER}_{COMPANY}_stock_summary.xlsx"

#     with pd.ExcelWriter(out_path, engine="xlsxwriter") as writer:
#         quarterly.round(4).to_excel(writer, sheet_name="Quarterly")
#         yearly_snapshot.round(4).to_excel(writer, sheet_name="Yearly_TTM")
#         q.to_excel(writer, sheet_name="Inputs (Merged Q)")

#     print(f"✅ 저장 완료: {out_path}")


# # ========== 전체 루프 실행 ==========
# for ticker, (company, fy_month) in companies.items():
#     analyze_company(ticker, company, fy_month)


In [ ]:
# import pandas as pd
# import numpy as np
# from pathlib import Path
# import glob
# from datetime import date

# # ====== 설정: 분석할 종목 목록 ======
# companies = {
#     "GOOG": ("alphabet", 12),
#     "BN": ("brookfield", 12),
#     "OXY": ("occidental petroleum", 12),
#     "CRSP": ("crispr therapeutics ag", 12),
#     "NVO": ("novo nordisk", 12),
#     "PATH": ("uipath", 12),
#     "OKTA": ("okta", 12),
#     "ILMN": ("illumina", 12),
#     "TSM": ("taiwan semiconductor manufacturing", 12),
#     "NTRA": ("natera", 12),
#     "META": ("meta platforms", 12),
#     "NU": ("nu holdings", 12),
#     "CPNG": ("coupang", 12),
#     "CVX": ("chevron", 12),
#     "PLTR": ("palantir technologies", 12),
#     "DDOG": ("datadog", 12),
#     "CRWD": ("crowdstrike", 12),
#     "AVGO": ("broadcom", 12),
#     "UNH": ("unitedhealth group", 12),
#     "LMND": ("lemonade", 12),
#     "AAPL": ("apple", 9),
#     "MITK": ("mitek systems", 12),
#     "MSFT": ("microsoft", 9),
#     "BBAI": ("bigbearai holdings", 12),
#     "VST": ("vistra", 12),
#     "VRT": ("vertiv holdings", 12),
#     "MP": ("mp materials", 12),
#     "COST": ("costco", 12),
#     "TEVA": ("teva pharmaceutical industries", 12),
#     "TSLA": ("tesla", 12),
#     "U": ("unity software", 12),
#     "RDDT": ("reddit", 12),
#     "SNOW": ("snowflake", 12),
#     "RKLB": ("rocket lab", 12),
#     "MELI": ("mercadolibre", 12),
#     "EH": ("ehang holdings", 12),
#     "GRAL": ("grail", 12),
#     "PLUG": ("plug power", 12),
#     "SE": ("sea", 12),
#     "TEM": ("tempus ai", 12),
#     "IREN": ("iren", 12),
#     "PI": ("impinj", 12),
#     "JOBY": ("joby aviation", 12),
#     "APP": ("applovin", 12),
#     "NVDA": ("nvidia", 1),
#     "COIN": ("coinbase gloabl", 12)
# }

# # ====== DCF 공통 가정 ======
# WACC = 0.10
# COE = 0.10
# G_SHORT = 0.05
# PROJ_YEARS = 5
# G_TERM = 0.025

# # ====== 기본 경로 ======
# BASE = Path(r"C:\Users\seung\OneDrive\주식")
# BACKTEST_DIR = BASE / "Back Test"
# FIN_DIR = BASE / "Financial_Data_real"
# SUMMARY_DIR = FIN_DIR / "Summary"
# SUMMARY_DIR.mkdir(parents=True, exist_ok=True)

# # ====== 공용 도우미 ======
# def safe_div(n: pd.Series, d: pd.Series, eps: float = 1e-9) -> pd.Series:
#     """안전한 나눗셈: |d|<eps 또는 NaN이면 NaN 반환"""
#     d_safe = d.copy()
#     d_safe = d_safe.where(d_safe.abs() > eps)
#     return n / d_safe

# def eps_growth_qoq(series: pd.Series, method: str = "classic", eps: float = 1e-9) -> pd.Series:
#     """
#     EPS QoQ 성장률(%)
#     - classic: (curr - prev) / prev * 100, 단 prev≈0이면 NaN
#     - midpoint: (curr - prev) / ((|curr|+|prev|)/2) * 100 → prev=0/음수에도 견고
#     - log: ln(curr/prev)*100, 단 curr<=0 or prev<=0이면 NaN
#     """
#     series = pd.Series(series).astype(float)
#     prev = series.shift(1)

#     if method == "classic":
#         growth = safe_div(series - prev, prev, eps) * 100.0
#     elif method == "midpoint":
#         denom = (series.abs() + prev.abs()) / 2.0
#         growth = safe_div(series - prev, denom, eps) * 100.0
#     elif method == "log":
#         mask_valid = (series > 0) & (prev > 0)
#         growth = pd.Series(np.nan, index=series.index, dtype=float)
#         growth[mask_valid] = np.log(series[mask_valid] / prev[mask_valid]) * 100.0
#     else:
#         raise ValueError("method must be 'classic', 'midpoint', or 'log'.")
#     return growth

# def dcf_equity_value_from_fcf(fcf0, net_debt, rate, g_short, years, g_term):
#     """FCF(FCFF/Owner)에서 Equity Value 계산 (2단계 성장 + 터미널)."""
#     if pd.isna(fcf0) or fcf0 <= 0:
#         return np.nan
#     pv = 0.0
#     for t in range(1, years + 1):
#         cf_t = fcf0 * ((1 + g_short) ** t)
#         pv += cf_t / ((1 + rate) ** t)
#     cf_N = fcf0 * ((1 + g_short) ** years)
#     tv = cf_N * (1 + g_term) / (rate - g_term)
#     pv_tv = tv / ((1 + rate) ** years)
#     ev = pv + pv_tv
#     return ev - (net_debt if pd.notna(net_debt) else 0.0)

# # ====== 개별 기업 분석 함수 ======
# def analyze_company(TICKER, COMPANY, FY_END_MONTH, MARKET_PRICE_OVERRIDE=None):
#     print(f"▶ 분석 시작: {TICKER} ({COMPANY})")

#     # --- 가격 CSV ---
#     company_folder = BACKTEST_DIR / COMPANY
#     price_candidates = glob.glob(str(company_folder / f"* {COMPANY} Historical Data.csv"))
#     if not price_candidates:
#         print(f"❌ 가격 CSV를 찾을 수 없음: {company_folder}\\* {COMPANY} Historical Data.csv")
#         return
#     PRICE_CSV = price_candidates[0]

#     # --- 재무 엑셀 ---
#     FIN_XLSX = FIN_DIR / f"{TICKER}_financials_Q.xlsx"
#     if not FIN_XLSX.exists():
#         print(f"❌ 재무 엑셀 없음: {FIN_XLSX}")
#         return

#     # --- 재무 시트 로드 ---
#     def load_financials(fin_path: Path):
#         income = pd.read_excel(fin_path, sheet_name="Income Statement", index_col=0)
#         balance = pd.read_excel(fin_path, sheet_name="Balance Sheet", index_col=0)
#         cashflw = pd.read_excel(fin_path, sheet_name="Cash Flow Statement", index_col=0)
#         ratios  = pd.read_excel(fin_path, sheet_name="Key Financial Ratios", index_col=0)

#         def wide_to_long(df):
#             df = df.copy()
#             df.index.name = "Metric"
#             out = df.T.reset_index().rename(columns={"index": "Date"})
#             out["Date"] = pd.to_datetime(out["Date"])
#             return out

#         return wide_to_long(income), wide_to_long(balance), wide_to_long(cashflw), wide_to_long(ratios)

#     income_q, balance_q, cash_q, ratios_q = load_financials(FIN_XLSX)

#     # --- 병합 ---
#     q = income_q.merge(balance_q, on="Date", how="left")\
#                 .merge(cash_q, on="Date", how="left")\
#                 .merge(ratios_q, on="Date", how="left")
#     q.sort_values("Date", inplace=True)

#     # --- 가격 매칭 ---
#     price = pd.read_csv(PRICE_CSV)
#     price.rename(columns=lambda c: c.strip(), inplace=True)
#     price["Date"] = pd.to_datetime(price["Date"])
#     price.sort_values("Date", inplace=True)

#     price_col = "Price" if "Price" in price.columns else [c for c in price.columns if "Price" in c][0]
#     px = price[["Date", price_col]].rename(columns={price_col: "Price"})
#     q = pd.merge_asof(q.sort_values("Date"), px, on="Date", direction="backward")
#     if MARKET_PRICE_OVERRIDE is not None:
#         q["Price"] = float(MARKET_PRICE_OVERRIDE)

#     # --- 분기 스냅샷 계산(필요 최소) ---
#     def compute_row_metrics(r):
#         get = lambda col, default=0.0: float(r[col]) if pd.notna(r.get(col)) else default
#         ebit = get("EBIT")
#         net_income = get("Net Income")
#         dep = get("Total Depreciation And Amortization - Cash Flow")
#         capex = -get("Net Change In Property, Plant, And Equipment")
#         delta_wc_cf = -get("Total Change In Assets/Liabilities")
#         price = get("Price")
#         shares_out = get("Shares Outstanding") or get("Basic Shares Outstanding")
#         market_cap = price * shares_out if shares_out else np.nan

#         fcff = (ebit * (1 - 0.25) + dep - capex - delta_wc_cf) if pd.notna(ebit) else np.nan
#         owner_earn = (net_income + dep - capex - delta_wc_cf) if pd.notna(net_income) else np.nan
#         return pd.Series({
#             "Price": price, "Market Cap": market_cap,
#             "FCFF": fcff, "Owner Earnings": owner_earn
#         })

#     q_metrics = q.apply(compute_row_metrics, axis=1)
#     quarterly = pd.concat([q[["Date"]], q_metrics], axis=1).set_index("Date")

#     # --- TTM 계산 ---
#     qq = q.set_index("Date").sort_index()
#     def ttm_series(col): return qq[col].rolling(4, min_periods=4).sum()

#     net_income_ttm = ttm_series("Net Income")
#     ebit_ttm = ttm_series("EBIT")
#     dep_ttm = ttm_series("Total Depreciation And Amortization - Cash Flow")
#     capex_ttm = (-qq["Net Change In Property, Plant, And Equipment"]).rolling(4, min_periods=4).sum()
#     delta_wc_ttm = (-qq["Total Change In Assets/Liabilities"]).rolling(4, min_periods=4).sum()
#     div_ps_ttm = (abs(qq["Common Stock Dividends Paid"]) / qq["Shares Outstanding"]).rolling(4, min_periods=4).sum()

#     snap = qq[["Total Liabilities","Cash On Hand","Shares Outstanding","Price"]].copy()

#     fcff_ttm = ebit_ttm * (1 - 0.25) + dep_ttm - capex_ttm - delta_wc_ttm
#     owner_ttm = net_income_ttm + dep_ttm - capex_ttm - delta_wc_ttm

#     # ---- PEG (표준: PER_TTM / EPS_TTM YoY%) ----
#     eps_ttm = net_income_ttm / snap["Shares Outstanding"]                     # 최근 4분기 합 EPS
#     pe_ttm = snap["Price"] / eps_ttm                                          # PER_TTM
#     eps_growth_yoy = eps_ttm.pct_change(4) * 100                              # EPS_TTM(현재)/EPS_TTM(1년전)-1
#     div_yield_ttm = (div_ps_ttm / snap["Price"]) * 100

#     eps_growth_yoy = (eps_ttm - eps_ttm.shift(4)) / eps_ttm.shift(4).abs() * 100
    
#     div_yield_ttm = (div_ps_ttm / snap["Price"]) * 100
    
#     peg_ttm = safe_div(pe_ttm, eps_growth_yoy)
#     pegy_ttm = safe_div(pe_ttm, eps_growth_yoy + div_yield_ttm)

#     # ---- DCF 적정가 (TTM FCF 이용) ----
#     net_debt_series = snap["Total Liabilities"] - snap["Cash On Hand"]
#     equity_from_fcff = pd.Series({dt: dcf_equity_value_from_fcf(fcff_ttm.loc[dt], net_debt_series.loc[dt], WACC, G_SHORT, PROJ_YEARS, G_TERM) for dt in qq.index})
#     equity_from_owner = pd.Series({dt: dcf_equity_value_from_fcf(owner_ttm.loc[dt], 0.0, COE, G_SHORT, PROJ_YEARS, G_TERM) for dt in qq.index})
#     dcf_price_fcff = equity_from_fcff / snap["Shares Outstanding"]
#     dcf_price_owner = equity_from_owner / snap["Shares Outstanding"]

#     yearly_ttm_full = pd.DataFrame({
#         "Price": snap["Price"],
#         "EPS_TTM": eps_ttm, "PE_TTM": pe_ttm, "EPS_Growth_YoY_%": eps_growth_yoy,
#         "PEG_TTM": peg_ttm, "PEGY_TTM": pegy_ttm,
#         "DCF_FCFF_Price": dcf_price_fcff,
#         "DCF_OwnerEarnings_Price": dcf_price_owner
#     })

#     # ---- 분기(단기) PEG도 참고용 제공: EPS QoQ 기준(안전 계산) ----
#     eps_q = qq["EPS - Earnings Per Share"]
#     eps_q_growth_qoq = (eps_q - eps_q.shift(1)) / eps_q.shift(1).abs() * 100

#     price_q = qq["Price"]
#     pe_q = price_q / eps_q
#     div_q = abs(qq["Common Stock Dividends Paid"])
#     shares_out_q = qq["Shares Outstanding"]
#     div_ps_q = div_q / shares_out_q
#     div_yield_q = div_ps_q / price_q * 100
    
#     peg_q = safe_div(pe_q, eps_q_growth_qoq)
#     pegy_q = safe_div(pe_q, eps_q_growth_qoq + div_yield_q)
    
#     quarterly["EPS_Q"] = eps_q
#     quarterly["PE_Q"] = pe_q
#     quarterly["EPS_Growth_QoQ_%"] = eps_q_growth_qoq
#     quarterly["PEG_Q"] = peg_q
#     quarterly["PEGY_Q"] = pegy_q


#     # ---- 회계연도 Q4(마감 분기) 스냅샷만 Yearly_TTM로 저장 ----
#     freq_map = {12:'Q-DEC',11:'Q-NOV',10:'Q-OCT',9:'Q-SEP',8:'Q-AUG',7:'Q-JUL',6:'Q-JUN',5:'Q-MAY',4:'Q-APR',3:'Q-MAR',2:'Q-FEB',1:'Q-JAN'}
#     fq_period = yearly_ttm_full.index.to_period(freq_map[FY_END_MONTH])
#     yearly_snapshot = yearly_ttm_full[(fq_period.quarter == 4)].copy()

#     # ---- 저장 ----
#     today_str = date.today().isoformat()
#     out_path = SUMMARY_DIR / f"{today_str}_{TICKER}_{COMPANY}_stock_summary.xlsx"
#     with pd.ExcelWriter(out_path, engine="xlsxwriter") as writer:
#         quarterly.round(4).to_excel(writer, sheet_name="Quarterly")
#         yearly_snapshot.round(4).to_excel(writer, sheet_name="Yearly_TTM")
#         q.to_excel(writer, sheet_name="Inputs (Merged Q)")
#     print(f"✅ 저장 완료: {out_path}")

# # ========== 전체 루프 실행 ==========
# for ticker, (company, fy_month) in companies.items():
#     try:
#         analyze_company(ticker, company, fy_month)
#     except Exception as e:
#         print(f"⚠️ {ticker} 처리 중 에러: {e}")


In [ ]:
# 통합 코드

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import glob
from datetime import date

# ====== 설정: 분석할 종목 목록 ======
companies = {
    "GOOG": ("alphabet", 12),
    "BN": ("brookfield", 12),
    "OXY": ("occidental petroleum", 12),
    "CRSP": ("crispr therapeutics ag", 12),
    "NVO": ("novo nordisk", 12),
    "PATH": ("uipath", 12),
    "OKTA": ("okta", 12),
    "ILMN": ("illumina", 12),
    "TSM": ("taiwan semiconductor manufacturing", 12),
    "NTRA": ("natera", 12),
    "META": ("meta platforms", 12),
    "NU": ("nu holdings", 12),
    "CPNG": ("coupang", 12),
    "CVX": ("chevron", 12),
    "PLTR": ("palantir technologies", 12),
    "DDOG": ("datadog", 12),
    "CRWD": ("crowdstrike", 12),
    "AVGO": ("broadcom", 12),
    "UNH": ("unitedhealth group", 12),
    "LMND": ("lemonade", 12),
    "AAPL": ("apple", 9),
    "MITK": ("mitek systems", 12),
    "MSFT": ("microsoft", 9),
    "BBAI": ("bigbearai holdings", 12),
    "VST": ("vistra", 12),
    "VRT": ("vertiv holdings", 12),
    "MP": ("mp materials", 12),
    "COST": ("costco", 12),
    "TEVA": ("teva pharmaceutical industries", 12),
    "TSLA": ("tesla", 12),
    "U": ("unity software", 12),
    "RDDT": ("reddit", 12),
    "SNOW": ("snowflake", 12),
    "RKLB": ("rocket lab", 12),
    "MELI": ("mercadolibre", 12),
    "EH": ("ehang holdings", 12),
    "GRAL": ("grail", 12),
    "PLUG": ("plug power", 12),
    "SE": ("sea", 12),
    "TEM": ("tempus ai", 12),
    "IREN": ("iren", 12),
    "PI": ("impinj", 12),
    "JOBY": ("joby aviation", 12),
    "APP": ("applovin", 12),
    "NVDA": ("nvidia", 1),
    "SBGSY": ("schneider electric se",12),
    "COIN": ("coinbase gloabl", 12)
}

# ====== DCF 공통 가정 ======
WACC = 0.10
COE = 0.10
G_SHORT = 0.05
PROJ_YEARS = 5
G_TERM = 0.025

# ====== 기본 경로 ======
BASE = Path(r"C:\Users\seung\OneDrive\주식")
BACKTEST_DIR = BASE / "Back Test"
FIN_DIR = BASE / "Financial_Data_real"
SUMMARY_DIR = FIN_DIR / "Summary"
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)

# ====== 공용 도우미 ======
def safe_div(n: pd.Series, d: pd.Series, eps: float = 1e-9) -> pd.Series:
    d_safe = d.copy().where(d.abs() > eps)
    return n / d_safe

def dcf_equity_value_from_fcf(fcf0, net_debt, rate, g_short, years, g_term):
    if pd.isna(fcf0) or fcf0 <= 0:
        return np.nan
    pv = sum(fcf0 * (1 + g_short)**t / (1 + rate)**t for t in range(1, years+1))
    cf_N = fcf0 * (1 + g_short)**years
    tv = cf_N * (1 + g_term) / (rate - g_term)
    pv_tv = tv / (1 + rate)**years
    return pv + pv_tv - (net_debt if pd.notna(net_debt) else 0)

def pct_change_abs(series, periods=1):
    prev = series.shift(periods)
    return (series - prev) / prev.abs() * 100

# ====== 동일 내용 반복 함수 ======
def analyze_company(TICKER, COMPANY, FY_END_MONTH, MARKET_PRICE_OVERRIDE=None):
    print(f"\n▶ 분석 시작: {TICKER} ({COMPANY})")

    # -- 가격 CSV & 재무 XLSX 경로 --
    company_folder = BACKTEST_DIR / COMPANY
    price_candidates = glob.glob(str(company_folder / f"* {COMPANY} Historical Data.csv"))
    if not price_candidates:
        print(f"❌ 가격 CSV 없음: {COMPANY}")
        return
    PRICE_CSV = price_candidates[0]

    FIN_XLSX = FIN_DIR / f"{TICKER}_financials_Q.xlsx"
    if not FIN_XLSX.exists():
        print(f"❌ 재무 엑셀 없음: {TICKER}")
        return

    # -- 재무 시트 로드 --
    def load_financials(fp):
        df = lambda sh: pd.read_excel(fp, sheet_name=sh, index_col=0).T.reset_index().rename(columns={"index":"Date"}).assign(Date=lambda x: pd.to_datetime(x["Date"]))
        return df("Income Statement"), df("Balance Sheet"), df("Cash Flow Statement"), df("Key Financial Ratios")

    inc, bal, cf, rat = load_financials(FIN_XLSX)
    q = inc.merge(bal, on="Date", how="left").merge(cf, on="Date", how="left").merge(rat, on="Date", how="left")
    q.sort_values("Date", inplace=True)

    # -- 가격 매칭 --
    price = pd.read_csv(PRICE_CSV); price["Date"] = pd.to_datetime(price["Date"])
    price.rename(columns=lambda c: c.strip(), inplace=True)
    price.sort_values("Date", inplace=True)
    price_col = next(c for c in price.columns if "Price" in c)
    px = price[["Date", price_col]].rename(columns={price_col: "Price"})
    q = pd.merge_asof(q, px, on="Date", direction="backward")
    if MARKET_PRICE_OVERRIDE is not None:
        q["Price"] = float(MARKET_PRICE_OVERRIDE)

    # -- 분기별 계산 --
    get = lambda r, c, d=0.0: float(r[c]) if pd.notna(r.get(c)) else d
    def calc_q(r):
        pe_share = get(r, "EPS - Earnings Per Share") or get(r, "Basic EPS")
        pr = get(r, "Price"); sh = get(r, "Shares Outstanding") or get(r, "Basic Shares Outstanding")
        ncav = get(r, "Total Current Assets") - get(r, "Total Liabilities")
        eps = (get(r, "Net Income") / sh) if sh else np.nan
        graham = np.sqrt(22.5 * max(eps,0) * max((get(r, "Share Holder Equity")/sh) if sh else np.nan,0)) if sh else np.nan
        div_ps = abs(get(r, "Common Stock Dividends Paid")); div_yield = (div_ps/sh)/pr*100 if sh and pr else np.nan

        return pd.Series({
            "Price": pr,
            "Market Cap": pr*sh if sh else np.nan,
            "NCAV": ncav,
            "Undervalued by NCAV": pr*sh < ncav if sh else np.nan,
            "Graham Number": graham,
            "FCFF": (get(r,"EBIT")*0.75 + get(r,"Total Depreciation And Amortization - Cash Flow") - (-get(r,"Net Change In Property, Plant, And Equipment")) - (-get(r,"Total Change In Assets/Liabilities"))) if pd.notna(get(r,"EBIT")) else np.nan,
            "Owner Earnings": (get(r,"Net Income") + get(r,"Total Depreciation And Amortization - Cash Flow") + get(r,"Net Change In Property, Plant, And Equipment") + get(r,"Total Change In Assets/Liabilities")) if pd.notna(get(r,"Net Income")) else np.nan,
            "DDM Value": (div_ps*(1+G_SHORT)/(COE-G_SHORT)) if div_ps else np.nan
        })

    quarterly = pd.concat([q[["Date"]], q.apply(calc_q, axis=1)], axis=1).set_index("Date")

    # -- QoQ PEG 계산 --
    eps_q = q["EPS - Earnings Per Share"]
    eps_q_g = (eps_q - eps_q.shift(1)) / eps_q.shift(1).abs() * 100
    pe_q = q["Price"] / eps_q
    div_yield_q = (abs(q["Common Stock Dividends Paid"])/q["Shares Outstanding"])/q["Price"]*100
    quarterly["PEG_Q"] = safe_div(pe_q, eps_q_g)
    quarterly["PEGY_Q"] = safe_div(pe_q, eps_q_g + div_yield_q)

    # -- TTM 계산 --
    qq = q.set_index("Date").sort_index()
    ttm = lambda col: qq[col].rolling(4, min_periods=4).sum()
    net = ttm("Net Income"); ebit = ttm("EBIT"); dep = ttm("Total Depreciation And Amortization - Cash Flow")
    capex = (-qq["Net Change In Property, Plant, And Equipment"]).rolling(4).sum()
    delta_wc = (-qq["Total Change In Assets/Liabilities"]).rolling(4).sum()
    div_ps = (abs(qq["Common Stock Dividends Paid"])/qq["Shares Outstanding"]).rolling(4).sum()
    snap = qq[["Total Liabilities","Cash On Hand","Shares Outstanding","Total Current Assets","Total Assets","Price"]]

    ncav_ttm = snap["Total Current Assets"] - snap["Total Liabilities"]
    market_cap_ttm = snap["Price"] * snap["Shares Outstanding"]
    roic = ebit / (snap["Total Current Assets"] + (snap["Total Assets"] - snap["Total Current Assets"]))
    ev = market_cap_ttm + snap["Total Liabilities"] - snap["Cash On Hand"]
    ev_ebit = ev / ebit

    eps_ttm = net / snap["Shares Outstanding"]
    pe_ttm = snap["Price"] / eps_ttm
    eps_yoy = eps_ttm.pct_change(4)*100
    eps_yoy = pct_change_abs(eps_ttm, periods=4)
    div_yield_ttm = div_ps / snap["Price"] * 100
    peg_ttm = safe_div(pe_ttm, eps_yoy)
    pegy_ttm = safe_div(pe_ttm, eps_yoy + div_yield_ttm)

    # DCF
    net_debt = snap["Total Liabilities"] - snap["Cash On Hand"]
    dcf_fcff = pd.Series({dt: dcf_equity_value_from_fcf((ebit.loc[dt]*0.75 + dep.loc[dt] - capex.loc[dt] - delta_wc.loc[dt]), net_debt.loc[dt], WACC, G_SHORT, PROJ_YEARS, G_TERM) for dt in snap.index})
    dcf_owner = pd.Series({dt: dcf_equity_value_from_fcf((net.loc[dt] + dep.loc[dt] - capex.loc[dt] - delta_wc.loc[dt]), 0.0, COE, G_SHORT, PROJ_YEARS, G_TERM) for dt in snap.index})
    dcf_fcff_price = dcf_fcff / snap["Shares Outstanding"]
    dcf_owner_price = dcf_owner / snap["Shares Outstanding"]

    yearly = pd.DataFrame({
        "Market Cap": market_cap_ttm,
        "NCAV_TTM": ncav_ttm,
        "Undervalued by NCAV": market_cap_ttm < ncav_ttm,
        "ROIC": roic,
        "EV/EBIT": ev_ebit,
        "EPS_TTM": eps_ttm, "PE_TTM": pe_ttm, "EPS_Growth_YoY_%": eps_yoy,
        "PEG_TTM": peg_ttm, "PEGY_TTM": pegy_ttm,
        "Graham Number": quarterly["Graham Number"],
        "DDM Value": quarterly["DDM Value"],
        "DCF_FCFF_Price": dcf_fcff_price,
        "DCF_OwnerEarnings_Price": dcf_owner_price,
        "Altman Z": (1.2*((snap["Total Current Assets"]-snap["Total Liabilities"])/snap["Total Assets"]) +
                     1.4*((qq["Retained Earnings (Accumulated Deficit)"])/snap["Total Assets"]) +
                     3.3*(ebit/snap["Total Assets"]) +
                     0.6*(market_cap_ttm/snap["Total Liabilities"]) +
                     1.0*(net/snap["Total Assets"]))
    })

    # Select year-end snapshot
    fq_map = {12:'Q-DEC',9:'Q-SEP',1:'Q-JAN'}
    fq = yearly.index.to_period(fq_map[FY_END_MONTH]).quarter
    snapshot = yearly[fq == 4]

    # -- Excel 저장 --
    today = date.today().isoformat()
    out = SUMMARY_DIR / f"{today}_{TICKER}_{COMPANY}_stock_summary.xlsx"
    with pd.ExcelWriter(out, engine="xlsxwriter") as w:
        quarterly.round(4).to_excel(w, sheet_name="Quarterly")
        snapshot.round(4).to_excel(w, sheet_name="Yearly_TTM")
        q.to_excel(w, sheet_name="Inputs")
    print(f"✅ 저장됨: {out}")

# ========== 전체 루프 실행 ==========
for t, (c, fy) in companies.items():
    try:
        analyze_company(t, c, fy)
    except Exception as e:
        print(f"⚠ {t} 에러: {e}")
